# 언론사 크롤링 가능성 테스트

Google News RSS로 검색한 기사의 본문을 정확히 가져올 수 있는 언론사를 추리는 노트북입니다. 위에서부터 순서대로 실행하면 됩니다.

**단계**
1. Google News RSS 검색 (검색어 × 언론사 × 날짜)
2. 언론사별 표본 후보 추출 → 표본에 필요한 만큼만 원문 URL 복원
3. 원문 HTML 다운로드
4. 저장된 HTML로 본문 추출 + 페이지 판정 (영상 클립 / 라이브 블로그 / 페이월 / 차단 / 본문 부족 …)
5. 언론사별 요약 → 판정별 언론사 딕셔너리 생성

**캐시**: RSS 검색·URL 복원 결과는 `source_test/`에 모든 회차가 같이 쓰고,
HTML과 판정 결과는 회차별 폴더(`source_test/round1/`, `source_test/round2/` …)에 저장됩니다.
다시 실행하면 이미 받은 RSS / URL / HTML은 재요청하지 않고 파일에서 읽어옵니다.
4단계 판정 기준(임계값, 패턴)만 바꿔서 다시 돌릴 때는 네트워크 요청이 전혀 발생하지 않습니다.

In [1]:
import gzip
import hashlib
import json
import math
import random
import re
import time
from collections import defaultdict
from datetime import datetime, timedelta, timezone
from email.utils import parsedate_to_datetime
from pathlib import Path
from urllib.parse import quote_plus, urlparse
from zoneinfo import ZoneInfo

import feedparser
import pandas as pd
import requests
from dateutil import parser as date_parser
from googlenewsdecoder import gnews_decoder_async
from lxml import html as lxml_html
from trafilatura import extract
from trafilatura.utils import decode_file

## 0. 설정

In [2]:
# ============================================================
# 테스트 대상
# ============================================================

# 2회차 테스트 대상 (1회차에서 크롤링 가능성이 남은 언론사)
CANDIDATE_SOURCES = {
    # 미국 증시/금융 전문
    "cnbc.com": "CNBC",

    # 경제/비즈니스 종합
    "fortune.com": "Fortune",
    "businessinsider.com": "Business Insider",
    "cnn.com": "CNN Business",
    "foxbusiness.com": "Fox Business",
    "cbsnews.com": "CBS News",
    "nypost.com": "New York Post",
    "usatoday.com": "USA Today",
    "theguardian.com": "The Guardian",

    # 투자 전문
    "fool.com": "The Motley Fool",
}

# 1회차(round1) 결과로 제외한 후보와 이유
EXCLUDED_SOURCES = {
    # 거의 전부 봇 차단(BLOCKED) 또는 HTTP 403
    "marketwatch.com": "MarketWatch",
    "barrons.com": "Barron's",
    "apnews.com": "Associated Press",
    "benzinga.com": "Benzinga",
    "zacks.com": "Zacks",
    "bloomberg.com": "Bloomberg",
    "wsj.com": "Wall Street Journal",
    "reuters.com": "Reuters",
    "axios.com": "Axios",
    "investing.com": "Investing.com",
    "ft.com": "Financial Times",

    # 차단이 많고 성공 0~1건
    "seekingalpha.com": "Seeking Alpha",
    "thestreet.com": "TheStreet",
    "forbes.com": "Forbes",

    # 뉴스 성격이 아님 (재테크 가이드, 보도자료·타사 기사 모음)
    "investopedia.com": "Investopedia",
    "kiplinger.com": "Kiplinger",
    "nasdaq.com": "Nasdaq",
    "finance.yahoo.com": "Yahoo Finance",
}

# 미국 뉴스 → 다음 날 한국 반도체주 시초가 분석에 맞춘 검색어
# (1회차 검색어 stock market, Wall Street, Federal Reserve, earnings, NVIDIA와 겹치지 않게 구성)
TEST_QUERIES = [
    # 시장
    "Dow Jones",
    "S&P 500",
    "Nasdaq",

    # 거시
    "interest rates",
    "inflation",
    "Treasury yields",
    "tariffs",

    # 반도체
    "semiconductor",
    "chip stocks",
    "memory chips",
    "Micron",
    "Samsung",
]

# 1회차 날짜(09-29, 10-14, 10-21)와 겹치지 않게, 요일을 섞어 9~10월에 고르게 배치
TEST_DATES = [
    "2026-09-08",
    "2026-09-18",
    "2026-10-02",
    "2026-10-07",
    "2026-10-16",
    "2026-10-26",
]

NEWS_TIMEZONE = ZoneInfo("America/New_York")

# ============================================================
# 캐시 경로
# ============================================================

CACHE_DIR = Path("source_test")

# RSS 검색·URL 복원 결과는 같은 검색이면 바뀌지 않으므로 모든 회차가 공유
RSS_CACHE_PATH = CACHE_DIR / "rss_cache.jsonl"
RESOLVE_CACHE_PATH = CACHE_DIR / "resolve_cache.jsonl"

# HTML과 판정 결과는 회차별 폴더에 따로 저장 ("round1"로 바꾸면 1회차 결과를 다시 볼 수 있음)
ROUND_NAME = "round2"
ROUND_DIR = CACHE_DIR / ROUND_NAME

HTML_DIR = ROUND_DIR / "html"
FETCH_CACHE_PATH = ROUND_DIR / "fetch_cache.jsonl"

PAGE_RESULT_PATH = ROUND_DIR / "page_results.parquet"
SOURCE_SUMMARY_PATH = ROUND_DIR / "source_summary.csv"
SOURCE_LISTS_PATH = ROUND_DIR / "source_lists.json"
REVIEW_SAMPLE_PATH = ROUND_DIR / "review_samples.csv"

# ============================================================
# 요청 설정
# ============================================================

REQUEST_TIMEOUT = 15
RSS_SLEEP_SEC = 1.0
FETCH_SLEEP_SEC = 0.5

REQUEST_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 "
        "(Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 "
        "(KHTML, like Gecko) "
        "Chrome/140.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
    "Accept-Language": "en-US,en;q=0.9",
}

# ============================================================
# 샘플링 / 판정 기준
# ============================================================

SAMPLE_PER_SOURCE = 60      # 언론사당 크롤링할 최대 기사 수
SAMPLE_SEED = 42
RESOLVE_BUFFER_RATIO = 0.2  # 복원 실패·도메인 불일치·중복 대비, 부족한 개수보다 20% 더 복원

MIN_BODY_WORDS = 80         # 본문 최소 단어 수 (짧은 단신 기사도 통과하도록 낮게)
PARAGRAPH_MIN_WORDS = 15    # 참고용 문단 수 계산 시, 이 단어 수 이상인 줄만 문단으로 인정
MIN_LD_COVERAGE = 0.6       # JSON-LD articleBody 대비 추출 본문 비율 하한

MIN_ARTICLE_PAGES = 20      # 판정에 필요한 최소 기사 페이지 수
USABLE_RATE = 0.8
UNSTABLE_RATE = 0.5
VIDEO_HEAVY_SHARE = 0.3     # 영상/라이브 페이지 비율이 이 이상이면 별도 표시

# ============================================================
# 페이지 유형 판별 패턴
# ============================================================

# URL 경로에 이 구간이 있으면 영상/오디오/갤러리 페이지
VIDEO_PATH_PATTERN = re.compile(
    r"/(video|videos|watch|clip|clips|v|live-tv|livetv|podcast|podcasts|audio|gallery|galleries|slideshow|photos)(/|$)",
    re.IGNORECASE,
)

# 호스트가 video.xxx.com 형태인 경우
VIDEO_HOST_PATTERN = re.compile(r"^(video|videos|watch|tv)\.", re.IGNORECASE)

LIVE_BLOG_PATH_PATTERN = re.compile(
    r"/(live|live-news|live-updates|liveblog)(/|$)|live-updates",
    re.IGNORECASE,
)

ARTICLE_LD_TYPES = {
    "Article",
    "NewsArticle",
    "ReportageNewsArticle",
    "AnalysisNewsArticle",
    "OpinionNewsArticle",
    "BackgroundNewsArticle",
    "ReviewNewsArticle",
    "BlogPosting",
    "Report",
}

PAYWALL_PHRASES = [
    "subscribe to continue",
    "subscribe to read",
    "subscription required",
    "to continue reading",
    "already a subscriber",
    "sign in to continue",
    "sign in to read",
    "log in to continue",
    "create a free account to continue",
    "this article is for subscribers",
    "this content is for subscribers",
    "unlock this article",
    "become a member to read",
]

# 봇 차단 / 챌린지 페이지 흔적 (<title> 또는 HTML 원문에서 검사)
BLOCK_TITLE_PHRASES = [
    "access denied",
    "just a moment",
    "attention required",
    "are you a robot",
    "pardon our interruption",
    "request unsuccessful",
    "verify you are human",
    "403 forbidden",
]

BLOCK_HTML_MARKERS = [
    "px-captcha",
    "captcha-delivery.com",
    "cf-chl-",
    "_incapsula_resource",
]

# ============================================================
# 기사 작성/수정 시각 추출 위치 (위에서부터 먼저 찾은 값 사용)
#   "ld:키"        → JSON-LD 기사 객체의 필드
#   "property:키"  → <meta property="키" content="...">
#   "name:키"      → <meta name="키" content="...">
#   "itemprop:키"  → <meta itemprop="키" content="...">
# ============================================================

PUBLISHED_DATE_FIELDS = [
    "ld:datePublished",
    "property:article:published_time",
    "itemprop:datePublished",
    "name:parsely-pub-date",
    "name:pubdate",
    "name:publish-date",
    "name:sailthru.date",
]

MODIFIED_DATE_FIELDS = [
    "ld:dateModified",
    "property:article:modified_time",
    "property:og:updated_time",
    "itemprop:dateModified",
]

# ============================================================
# 수집 대상이 아닌 페이지 판별
# ============================================================

# 언론사별 제외 구간: "호스트/경로" 문자열(www. 제거)에 정규식 검색
# URL만으로 판단하므로 다운로드 결과(차단, HTTP 오류 등)보다 먼저 적용
SECTION_EXCLUDE_RULES = {
    "businessinsider.com": [
        r"^markets\.",   # 보도자료 서브도메인
        r"^africa\.",    # 지역판 (본판 기사 복사본)
    ],
    "usatoday.com": [
        r"^ftw\.",           # 스포츠 팬 사이트
        r"^patriotswire\.",
        r"/story/shopping/",  # 쇼핑 할인 페이지 (HTTP 402 응답)
        r"/press-release/",   # 보도자료 구역
    ],
    "cnbc.com": [
        r"/investingclub/",   # 유료 투자 클럽 (Jim Cramer 포트폴리오 논평·매매 의견)
    ],
    "cnn.com": [
        r"^transcripts\.",   # 방송 대본
        r"/cnn-underscored/", # 쇼핑/제품 리뷰 (봇 차단 응답)
        r"^cnnicevents\.",   # 행사 페이지
    ],
    "nypost.com": [
        r"/betting/",     # 도박 프로모션
    ],
    "forbes.com": [
        r"/councils/",    # 유료 회원 기고
    ],
}

# 보도자료 배포사 (JSON-LD 원 출처 이름에 포함되면 보도자료)
PRESS_RELEASE_PROVIDERS = [
    "globenewswire",
    "globe newswire",
    "pr newswire",
    "prnewswire",
    "business wire",
    "accesswire",
    "access newswire",
    "newsfile",
    "eqs news",
]

# 본문 앞부분에 배포사 표기가 있으면 보도자료
PRESS_RELEASE_BODY_PATTERN = re.compile(
    r"PR ?Newswire|Business ?Wire|Globe ?Newswire|ACCESS ?Newswire|ACCESSWIRE|Newsfile|EQS-News|/CNW/",
    re.IGNORECASE,
)
PRESS_RELEASE_BODY_CHARS = 600

VERDICT_ORDER = ["USABLE", "UNSTABLE", "NOT_USABLE", "INSUFFICIENT_SAMPLE", "NO_RSS_RESULT"]

# 아래 판정은 URL/메타데이터/날짜 규칙으로 걸러낼 수 있으므로 성공률 계산 시 분모에서 제외
FILTERABLE_STATUSES = {
    "VIDEO_PAGE",        # 영상 클립
    "LIVE_BLOG",         # 라이브 블로그
    "EXCLUDED_SECTION",  # 지역판, 팬 사이트, 프로모션, 유료 기고
    "PRESS_RELEASE",     # 보도자료 (공시는 외부 API로 별도 수집)
    "NOT_ARTICLE",       # 작성자 프로필, 시세·카테고리·기업 소개 페이지 등
    "OUT_OF_WINDOW",     # 언론사 작성일이 검색 대상일과 다름
}

# 영상/라이브 비중 판단용
VIDEO_STATUSES = {"VIDEO_PAGE", "LIVE_BLOG"}

## 0-1. 공통 유틸 (캐시 읽기/쓰기)

In [3]:
CACHE_DIR.mkdir(parents=True, exist_ok=True)
HTML_DIR.mkdir(parents=True, exist_ok=True)

# 다운로드한 HTML이 git에 올라가지 않도록 캐시 폴더 전체를 제외
(CACHE_DIR / ".gitignore").write_text("*\n")


def append_jsonl(path, record):
    with open(path, "a", encoding="utf-8") as f:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")


def load_jsonl(path):
    path = Path(path)

    if not path.exists():
        return []

    records = []

    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                records.append(json.loads(line))

    return records


def now_utc():
    return datetime.now(timezone.utc).isoformat()


def url_hash(url):
    return hashlib.sha1(url.encode("utf-8")).hexdigest()


def html_cache_path(url):
    return HTML_DIR / f"{url_hash(url)}.html.gz"


def host_of(url):
    host = (urlparse(url).hostname or "").lower()

    if host.startswith("www."):
        host = host[4:]

    return host


def host_matches_source(url, source):
    host = host_of(url)

    return host == source or host.endswith("." + source)


def to_source_dict(domains):
    """도메인 리스트 → {도메인: 언론사명} (CANDIDATE_SOURCES 순서 유지)"""
    domains = set(domains)

    return {
        domain: name
        for domain, name in CANDIDATE_SOURCES.items()
        if domain in domains
    }


def make_session():
    session = requests.Session()
    session.headers.update(REQUEST_HEADERS)

    return session

## 1. Google News RSS 검색 (캐시)

In [4]:
def build_rss_url(query, source, target_date):
    """
    target_date 앞뒤 하루를 포함해 검색하고
    실제 날짜 필터링은 Python에서 미국 동부시간 기준으로 다시 수행합니다.
    """
    search_after = target_date - timedelta(days=1)
    search_before = target_date + timedelta(days=1)

    search_query = (
        f'"{query}" '
        f"site:{source} "
        f"after:{search_after.isoformat()} "
        f"before:{search_before.isoformat()}"
    )

    url = (
        "https://news.google.com/rss/search"
        f"?q={quote_plus(search_query)}"
        "&hl=en-US"
        "&gl=US"
        "&ceid=US:en"
    )

    return url, search_query


def fetch_rss(query, source, target_date, session):
    url, search_query = build_rss_url(query, source, target_date)

    response = session.get(url, timeout=REQUEST_TIMEOUT)
    response.raise_for_status()

    feed = feedparser.parse(response.content)

    entries = []

    for entry in feed.entries:
        published_raw = entry.get("published")

        if not published_raw:
            continue

        try:
            published_dt = parsedate_to_datetime(published_raw)
            published_et = published_dt.astimezone(NEWS_TIMEZONE)

        except Exception:
            continue

        # [실제 수집 코드에 반영할 것]
        # RSS 시각은 날짜만 있는 경우(03:00:00으로 채워짐)가 절반 이상이고,
        # 전날 저녁 기사가 다음 날로 잡히는 경우도 있어 실제 작성 시각과 다를 수 있음.
        # → 실제 수집에서는 RSS 검색·필터 범위를 넉넉하게(앞뒤 하루 이상) 잡고,
        #   최종 포함 여부는 기사 HTML의 언론사 작성 시각(page_published_at_et)으로 판단할 것.
        # (이 테스트는 언론사별 크롤링 가능 여부만 보므로 RSS 날짜로 먼저 걸러도 무방)
        if published_et.date() != target_date:
            continue

        rss_source = entry.get("source") or {}

        entries.append({
            "title": entry.get("title"),
            "google_news_url": entry.get("link"),
            "rss_source_title": rss_source.get("title"),
            "rss_source_href": rss_source.get("href"),
            "published_at": published_dt.isoformat(),
            "published_at_et": published_et.isoformat(),
        })

    return {
        "search_query": search_query,
        # 날짜 필터 전 RSS 항목 수 (100이면 결과가 잘렸을 가능성)
        "total_entries": len(feed.entries),
        "entries": entries,
    }


def rss_cache_key(query, source, target_date):
    return f"{target_date}|{source}|{query}"


def collect_rss_cached(sources, queries, dates, retry_errors=True):
    cache = {}

    for record in load_jsonl(RSS_CACHE_PATH):
        cache[record["key"]] = record  # 같은 key는 마지막 기록이 우선

    combos = [
        (datetime.strptime(d, "%Y-%m-%d").date(), source, query)
        for d in dates
        for source in sources
        for query in queries
    ]

    todo = []

    for target_date, source, query in combos:
        cached = cache.get(rss_cache_key(query, source, target_date))

        if cached and (cached["status"] == "OK" or not retry_errors):
            continue

        todo.append((target_date, source, query))

    print(f"RSS 조합 {len(combos)}개 중 캐시 사용 {len(combos) - len(todo)}개, 새로 요청 {len(todo)}개")

    session = make_session()

    for i, (target_date, source, query) in enumerate(todo, start=1):
        key = rss_cache_key(query, source, target_date)

        record = {
            "key": key,
            "target_date": target_date.isoformat(),
            "source": source,
            "query": query,
            "fetched_at": now_utc(),
        }

        try:
            result = fetch_rss(query, source, target_date, session)
            record.update(status="OK", error=None, **result)

        except Exception as e:
            record.update(
                status="ERROR",
                error=f"{type(e).__name__}: {e}",
                search_query=None,
                total_entries=0,
                entries=[],
            )

        append_jsonl(RSS_CACHE_PATH, record)
        cache[key] = record

        print(
            f"[{i}/{len(todo)}] {target_date} | {source} | {query} "
            f"→ {record['status']} {len(record['entries'])}건"
        )

        time.sleep(RSS_SLEEP_SEC)

    session.close()

    return [
        cache[rss_cache_key(query, source, target_date)]
        for target_date, source, query in combos
    ]

In [5]:
rss_records = collect_rss_cached(
    sources=CANDIDATE_SOURCES,
    queries=TEST_QUERIES,
    dates=TEST_DATES,
)

# 검색 결과 1건 = 1행 (같은 기사가 여러 검색어에 걸리면 여러 행)
rss_df = pd.DataFrame([
    {
        "target_date": record["target_date"],
        "source": record["source"],
        "query": record["query"],
        **entry,
    }
    for record in rss_records
    for entry in record["entries"]
])

# google_news_url 기준 중복 제거, 걸린 검색어는 리스트로 보존
articles_df = (
    rss_df
    .groupby(["source", "google_news_url"], as_index=False)
    .agg(
        query=("query", "first"),
        matched_queries=("query", lambda x: ", ".join(sorted(set(x)))),
        title=("title", "first"),
        rss_source_title=("rss_source_title", "first"),
        published_at_et=("published_at_et", "first"),
        target_date=("target_date", "first"),
    )
)

print(f"RSS 검색 결과 {len(rss_df)}행 → 중복 제거 {len(articles_df)}건")

RSS 조합 720개 중 캐시 사용 720개, 새로 요청 0개
RSS 검색 결과 3660행 → 중복 제거 2456건


In [6]:
rss_stats_df = (
    pd.DataFrame(rss_records)
    .assign(
        hits=lambda d: d["entries"].str.len(),
        is_error=lambda d: d["status"] == "ERROR",
        is_truncated=lambda d: d["total_entries"] >= 100,
    )
    .groupby("source")
    .agg(
        rss_requests=("key", "size"),
        rss_errors=("is_error", "sum"),
        rss_truncated=("is_truncated", "sum"),
        rss_hits=("hits", "sum"),
    )
    .join(articles_df.groupby("source").size().rename("rss_articles"))
    .fillna({"rss_articles": 0})
    .astype(int)
    .reindex(list(CANDIDATE_SOURCES))
    .reset_index()
)

# ── 변수 정리: RSS 단계 ──
RSS_HIT_SOURCES = to_source_dict(
    rss_stats_df.loc[rss_stats_df["rss_articles"] > 0, "source"]
)
RSS_EMPTY_SOURCES = to_source_dict(
    rss_stats_df.loc[rss_stats_df["rss_articles"] == 0, "source"]
)
RSS_ERROR_KEYS = [
    record["key"]
    for record in rss_records
    if record["status"] == "ERROR"
]

print("RSS 결과 있음:", list(RSS_HIT_SOURCES.values()))
print("RSS 결과 없음:", list(RSS_EMPTY_SOURCES.values()))
print("RSS 요청 실패:", len(RSS_ERROR_KEYS), "건")

rss_stats_df

RSS 결과 있음: ['CNBC', 'Fortune', 'Business Insider', 'CNN Business', 'Fox Business', 'CBS News', 'New York Post', 'USA Today', 'The Guardian', 'The Motley Fool']
RSS 결과 없음: []
RSS 요청 실패: 0 건


,source,rss_requests,rss_errors,rss_truncated,rss_hits,rss_articles
0,cnbc.com,72,0,0,451,266
1,fortune.com,72,0,0,375,214
2,businessinsider.com,72,0,8,848,605
3,cnn.com,72,0,0,227,139
4,foxbusiness.com,72,0,0,194,165
5,cbsnews.com,72,0,0,188,165
6,nypost.com,72,0,0,95,65
7,usatoday.com,72,0,0,148,118
8,theguardian.com,72,0,0,192,164
9,fool.com,72,0,7,942,555


## 2. 표본 후보 추출 + 원문 URL 복원 (캐시)

RSS로 찾은 기사 전부가 아니라, 언론사별 표본 후보를 먼저 정한 뒤 **표본에 필요한 만큼만** 원문 URL을 복원합니다.
복원 실패·도메인 불일치·중복으로 60개가 모자라면 다음 후보를 더 복원해서 채웁니다.
(실제 수집 단계에서는 찾은 기사를 모두 써야 하므로 전부 복원해야 함)

In [7]:
async def resolve_urls_cached(
    google_urls,
    batch_size=40,
    concurrency=4,
    retry_failed=False,
):
    cache = {}

    for record in load_jsonl(RESOLVE_CACHE_PATH):
        cache[record["google_news_url"]] = record

    unique_urls = list(dict.fromkeys(google_urls))

    todo = [
        url
        for url in unique_urls
        if url not in cache
        or (retry_failed and cache[url]["status"] != "SUCCESS")
    ]

    print(f"URL {len(unique_urls)}개 중 캐시 사용 {len(unique_urls) - len(todo)}개, 새로 복원 {len(todo)}개")

    for start in range(0, len(todo), batch_size):
        batch = todo[start:start + batch_size]

        try:
            results = await gnews_decoder_async(
                batch,
                interval=0.2,
                timeout=15.0,
                concurrency=concurrency,
            )

            if isinstance(results, dict):
                results = [results]

        except Exception as e:
            error = f"{type(e).__name__}: {e}"
            results = [{"success": False, "message": error}] * len(batch)

        for url, result in zip(batch, results):
            record = {
                "google_news_url": url,
                "original_url": result.get("decoded_url") if result.get("success") else None,
                "status": "SUCCESS" if result.get("success") else "FAILED",
                "error": None if result.get("success") else result.get("message"),
                "resolved_at": now_utc(),
            }

            append_jsonl(RESOLVE_CACHE_PATH, record)
            cache[url] = record

        print(f"  {min(start + batch_size, len(todo))}/{len(todo)} 완료")

    return {url: cache[url] for url in unique_urls}


def order_candidates(group, rng):
    """
    한 언론사의 전체 후보를 추출 순서대로 정렬합니다.
    검색어별로 섞은 뒤 검색어를 번갈아 가며 꺼냄 (특정 검색어 결과만 몰리지 않도록 라운드로빈)
    """
    by_query = []

    for _, query_group in group.groupby("query"):
        rows = query_group.to_dict("records")
        rng.shuffle(rows)
        by_query.append(rows)

    ordered = []

    while any(by_query):
        for rows in by_query:
            if rows:
                ordered.append(rows.pop())

    return ordered


async def sample_and_resolve(
    articles_df,
    per_source=SAMPLE_PER_SOURCE,
    seed=SAMPLE_SEED,
    buffer_ratio=RESOLVE_BUFFER_RATIO,
):
    """
    언론사별 후보 순서를 정하고, 앞에서부터 필요한 만큼만 원문 URL을 복원해 최대 per_source개를 뽑습니다.
    복원 실패·도메인 불일치·중복으로 모자라면 다음 후보를 더 복원해서 채웁니다.
    """
    queues = {}

    candidates = articles_df.sort_values(["source", "query", "google_news_url"])

    for source, group in candidates.groupby("source", sort=False):
        # 언론사마다 별도 시드 사용: 한 언론사에 기사가 추가돼도
        # 다른 언론사의 샘플은 그대로 유지되어 HTML을 다시 받지 않음
        rng = random.Random(f"{seed}|{source}")
        queues[source] = order_candidates(group, rng)

    picked = {source: [] for source in queues}
    seen_urls = set()

    while True:
        # 아직 모자란 언론사만, 부족한 개수 + 여유분만큼 다음 후보를 꺼냄
        batch = {}

        for source, queue in queues.items():
            shortage = per_source - len(picked[source])

            if shortage <= 0 or not queue:
                continue

            take = min(math.ceil(shortage * (1 + buffer_ratio)), len(queue))
            batch[source] = [queue.pop(0) for _ in range(take)]

        if not batch:
            break

        resolve_map = await resolve_urls_cached(
            [article["google_news_url"] for rows in batch.values() for article in rows]
        )

        for source, rows in batch.items():
            for article in rows:
                resolved = resolve_map[article["google_news_url"]]
                url = resolved["original_url"]

                if resolved["status"] != "SUCCESS":
                    continue

                # 복원된 URL이 검색한 언론사 도메인이 아니거나, 이미 뽑은 기사면 제외
                if not host_matches_source(url, source) or url in seen_urls:
                    continue

                if len(picked[source]) < per_source:
                    seen_urls.add(url)
                    picked[source].append({**article, "original_url": url})

    return pd.DataFrame([article for rows in picked.values() for article in rows])

In [8]:
sample_df = await sample_and_resolve(articles_df)

print(f"샘플 기사 수: {len(sample_df)}")

(
    articles_df.groupby("source").size().rename("rss_articles").to_frame()
    .join(sample_df.groupby("source").size().rename("sampled"))
    .fillna(0)
    .astype(int)
)

URL 713개 중 캐시 사용 713개, 새로 복원 0개
샘플 기사 수: 600


,rss_articles,sampled
source,,
businessinsider.com,605,60
cbsnews.com,165,60
cnbc.com,266,60
cnn.com,139,60
fool.com,555,60
fortune.com,214,60
foxbusiness.com,165,60
nypost.com,65,60
theguardian.com,164,60


## 3. 원문 HTML 다운로드 (캐시)

In [9]:
def interleave_by_source(sample_df):
    """같은 사이트에 연속 요청하지 않도록 언론사를 번갈아 정렬"""
    queues = [
        group["original_url"].tolist()
        for _, group in sample_df.groupby("source")
    ]

    ordered = []

    while any(queues):
        for queue in queues:
            if queue:
                ordered.append(queue.pop(0))

    return ordered


def is_retryable_fetch(record):
    """일시적 오류(타임아웃, 429, 5xx)만 재시도 대상"""
    if record["fetch_status"] in {"TIMEOUT", "REQUEST_ERROR"}:
        return True

    return record["http_status"] in {429, 500, 502, 503, 504}


def fetch_html_cached(urls, retry_errors=False):
    cache = {}

    for record in load_jsonl(FETCH_CACHE_PATH):
        cache[record["url"]] = record

    todo = [
        url
        for url in urls
        if url not in cache
        or (retry_errors and is_retryable_fetch(cache[url]))
    ]

    print(f"URL {len(urls)}개 중 캐시 사용 {len(urls) - len(todo)}개, 새로 다운로드 {len(todo)}개")

    session = make_session()

    for i, url in enumerate(todo, start=1):
        record = {
            "url": url,
            "final_url": None,
            "http_status": None,
            "content_type": None,
            "fetch_status": None,
            "error": None,
            "html_saved": False,
            "fetched_at": now_utc(),
        }

        try:
            response = session.get(
                url,
                timeout=REQUEST_TIMEOUT,
                allow_redirects=True,
            )

        except requests.Timeout:
            record.update(fetch_status="TIMEOUT", error="Request timeout")

        except requests.RequestException as e:
            record.update(fetch_status="REQUEST_ERROR", error=f"{type(e).__name__}: {e}")

        else:
            content_type = response.headers.get("Content-Type", "").lower()

            record.update(
                final_url=response.url,
                http_status=response.status_code,
                content_type=content_type,
            )

            if response.status_code != 200:
                record["fetch_status"] = "HTTP_ERROR"
            elif "html" not in content_type:
                record["fetch_status"] = "NOT_HTML"
            else:
                record["fetch_status"] = "OK"

            # 오류 응답이라도 HTML이면 저장 (차단 페이지 확인용)
            if "html" in content_type:
                html_cache_path(url).write_bytes(gzip.compress(response.content))
                record["html_saved"] = True

        append_jsonl(FETCH_CACHE_PATH, record)
        cache[url] = record

        print(
            f"[{i}/{len(todo)}] {host_of(url)} | "
            f"{record['fetch_status']} ({record['http_status']})"
        )

        time.sleep(FETCH_SLEEP_SEC)

    session.close()

    return {url: cache[url] for url in urls}

In [10]:
# 일시적 오류(타임아웃, 연결 오류, 429, 5xx)로 캐시된 URL만 다시 요청
fetch_map = fetch_html_cached(interleave_by_source(sample_df), retry_errors=True)

URL 600개 중 캐시 사용 570개, 새로 다운로드 30개
[1/30] usatoday.com | OK (200)
[2/30] usatoday.com | HTTP_ERROR (500)
[3/30] cnn.com | OK (200)
[4/30] markets.businessinsider.com | OK (200)
[5/30] transcripts.cnn.com | OK (200)
[6/30] markets.businessinsider.com | OK (200)
[7/30] africa.businessinsider.com | OK (200)
[8/30] transcripts.cnn.com | OK (200)
[9/30] markets.businessinsider.com | OK (200)
[10/30] transcripts.cnn.com | OK (200)
[11/30] markets.businessinsider.com | OK (200)
[12/30] transcripts.cnn.com | OK (200)
[13/30] markets.businessinsider.com | OK (200)
[14/30] transcripts.cnn.com | OK (200)
[15/30] markets.businessinsider.com | OK (200)
[16/30] cnn.com | OK (200)
[17/30] africa.businessinsider.com | OK (200)
[18/30] edition.cnn.com | OK (200)
[19/30] usatoday.com | OK (200)
[20/30] markets.businessinsider.com | OK (200)
[21/30] cnnicevents.cnn.com | HTTP_ERROR (403)
[22/30] usatoday.com | OK (200)
[23/30] markets.businessinsider.com | OK (200)
[24/30] cnn.com | OK (200)
[25/30] usat

## 4. 본문 추출 + 페이지 판정 (저장된 HTML만 사용, 네트워크 요청 없음)

판정 기준(0. 설정의 임계값·패턴)을 바꾼 뒤 이 섹션부터 다시 실행하면 됩니다.

In [11]:
def read_cached_html(url):
    path = html_cache_path(url)

    if not path.exists():
        return None

    return decode_file(gzip.decompress(path.read_bytes()))


def parse_json_ld(tree):
    """페이지의 JSON-LD 최상위 객체(+@graph)를 평탄화해서 리스트로 반환"""
    items = []

    for node in tree.xpath('//script[@type="application/ld+json"]'):
        try:
            data = json.loads(node.text_content().strip())

        except Exception:
            continue

        stack = [data]

        while stack:
            current = stack.pop()

            if isinstance(current, list):
                stack.extend(current)

            elif isinstance(current, dict):
                items.append(current)

                if "@graph" in current:
                    stack.append(current["@graph"])

    return items


def ld_types_of(item):
    types = item.get("@type", [])

    if isinstance(types, str):
        types = [types]

    return {t for t in types if isinstance(t, str)}


def meta_content(tree, attr, value):
    found = tree.xpath(f'//meta[@{attr}="{value}"]/@content')

    return found[0].strip().lower() if found else None


def parse_bool(value):
    if isinstance(value, bool):
        return value

    if isinstance(value, str) and value.strip().lower() in {"true", "false"}:
        return value.strip().lower() == "true"

    return None


def to_news_tz(value):
    """
    날짜 문자열 → 미국 동부시간 datetime.
    시간대 표기가 없는 값은 동부시간으로 간주하고 tz_assumed=True로 표시합니다.
    """
    if not isinstance(value, str) or not value.strip():
        return None, False

    try:
        dt = date_parser.parse(value.strip())

    except (ValueError, OverflowError):
        return None, False

    tz_assumed = dt.tzinfo is None

    if tz_assumed:
        dt = dt.replace(tzinfo=NEWS_TIMEZONE)

    return dt.astimezone(NEWS_TIMEZONE), tz_assumed


def find_page_date(tree, article_items, fields):
    """fields 순서대로 찾아서 처음 해석되는 날짜를 (datetime, tz_assumed, 출처) 로 반환"""
    for field in fields:
        kind, key = field.split(":", 1)

        if kind == "ld":
            values = [item.get(key) for item in article_items]
        else:
            values = tree.xpath(f'//meta[@{kind}="{key}"]/@content')

        for value in values:
            dt, tz_assumed = to_news_tz(value)

            if dt:
                return dt, tz_assumed, field

    return None, False, None


def find_original_provider(article_items):
    """JSON-LD에 적힌 원 출처 이름 (provider, sourceOrganization, 기관 author 순)"""
    for item in article_items:
        candidates = [item.get("provider"), item.get("sourceOrganization"), item.get("author")]

        for value in candidates:
            if isinstance(value, list):
                value = value[0] if value else None

            if not isinstance(value, dict):
                continue

            if value.get("@type") in (None, "Organization", "NewsMediaOrganization") and value.get("name"):
                return value["name"]

    return None


def is_excluded_section(source, url):
    target = host_of(url) + urlparse(url).path

    return any(re.search(rule, target) for rule in SECTION_EXCLUDE_RULES.get(source, []))


def extract_page_features(raw_html, final_url):
    # lxml은 encoding 선언이 있는 str을 받지 못하므로 XML 선언 제거
    tree = lxml_html.fromstring(re.sub(r"^\s*<\?xml[^>]*\?>", "", raw_html))

    page_title = (tree.findtext(".//title") or "").strip()

    # ── 메타데이터 ──
    og_type = meta_content(tree, "property", "og:type")
    twitter_card = meta_content(tree, "name", "twitter:card")
    has_og_video = bool(tree.xpath('//meta[starts-with(@property, "og:video")]'))

    ld_items = parse_json_ld(tree)
    ld_types = set().union(*[ld_types_of(item) for item in ld_items]) if ld_items else set()

    article_items = [item for item in ld_items if ld_types_of(item) & ARTICLE_LD_TYPES]

    ld_body_words = max(
        (len(str(item.get("articleBody") or "").split()) for item in article_items),
        default=0,
    )

    ld_free_values = [
        parse_bool(item.get("isAccessibleForFree"))
        for item in article_items
        if "isAccessibleForFree" in item
    ]
    ld_is_free = None if not ld_free_values else all(v is not False for v in ld_free_values)

    original_provider = find_original_provider(article_items)

    # ── 언론사가 적은 작성/수정 시각 ──
    published_dt, published_tz_assumed, published_from = find_page_date(
        tree, article_items, PUBLISHED_DATE_FIELDS
    )
    modified_dt, modified_tz_assumed, modified_from = find_page_date(
        tree, article_items, MODIFIED_DATE_FIELDS
    )

    # ── URL 기반 ──
    path = urlparse(final_url).path

    url_is_video = bool(
        VIDEO_PATH_PATTERN.search(path)
        or VIDEO_HOST_PATTERN.search(host_of(final_url))
    )
    url_is_live = bool(LIVE_BLOG_PATH_PATTERN.search(path))

    # ── 차단 페이지 흔적 ──
    title_lower = page_title.lower()
    html_lower = raw_html[:50000].lower()

    is_blocked = (
        any(phrase in title_lower for phrase in BLOCK_TITLE_PHRASES)
        or any(marker in html_lower for marker in BLOCK_HTML_MARKERS)
    )

    # ── 본문 ──
    body = extract(
        raw_html,
        url=final_url,
        include_comments=False,
        include_tables=False,
        favor_precision=True,
    )
    body = (body or "").strip()

    body_words = len(body.split())
    body_paragraphs = sum(
        1
        for line in body.split("\n")
        if len(line.split()) >= PARAGRAPH_MIN_WORDS
    )

    body_lower = body.lower()
    paywall_phrase = next((p for p in PAYWALL_PHRASES if p in body_lower), None)

    provider_lower = (original_provider or "").lower()

    is_press_release = bool(
        any(name in provider_lower for name in PRESS_RELEASE_PROVIDERS)
        or PRESS_RELEASE_BODY_PATTERN.search(body[:PRESS_RELEASE_BODY_CHARS])
    )

    return {
        "page_title": page_title,
        "og_type": og_type,
        "twitter_card": twitter_card,
        "has_og_video": has_og_video,
        "ld_types": sorted(ld_types),
        "ld_has_article": bool(ld_types & ARTICLE_LD_TYPES),
        "ld_has_video": "VideoObject" in ld_types,
        "ld_has_liveblog": "LiveBlogPosting" in ld_types,
        "ld_body_words": ld_body_words,
        "ld_is_free": ld_is_free,
        "original_provider": original_provider,
        "is_press_release": is_press_release,
        "page_published_at_et": published_dt.isoformat() if published_dt else None,
        "page_published_from": published_from,
        "page_modified_at_et": modified_dt.isoformat() if modified_dt else None,
        "page_modified_from": modified_from,
        "page_date_tz_assumed": published_tz_assumed or modified_tz_assumed,
        "url_is_video": url_is_video,
        "url_is_live": url_is_live,
        "is_blocked": is_blocked,
        "paywall_phrase": paywall_phrase,
        "body": body,
        "body_words": body_words,
        "body_paragraphs": body_paragraphs,
        "ld_coverage": (body_words / ld_body_words) if ld_body_words else None,
    }


def classify_page(source, target_date, fetch, features):
    if fetch is None:
        return "NOT_FETCHED"

    # 제외 구역은 URL만으로 판단 (실제 수집에서는 다운로드 전에 거를 수 있음)
    if is_excluded_section(source, fetch["final_url"] or fetch["url"]):
        return "EXCLUDED_SECTION"

    if fetch["fetch_status"] in {"TIMEOUT", "REQUEST_ERROR", "NOT_HTML"}:
        return fetch["fetch_status"]

    if features is not None and features["is_blocked"]:
        return "BLOCKED"

    if fetch["fetch_status"] == "HTTP_ERROR":
        return f"HTTP_{fetch['http_status']}"

    if not host_matches_source(fetch["final_url"], source):
        return "REDIRECTED_AWAY"

    if features is None:
        return "NOT_FETCHED"

    # 영상 클립: URL이 영상 경로이거나, 메타데이터상 기사가 아니라 영상인 페이지
    if (
        features["url_is_video"]
        or (features["og_type"] or "").startswith("video")
        or (features["ld_has_video"] and not features["ld_has_article"])
    ):
        return "VIDEO_PAGE"

    if features["url_is_live"] or features["ld_has_liveblog"]:
        return "LIVE_BLOG"

    if features["is_press_release"]:
        return "PRESS_RELEASE"

    # 기사 타입 메타데이터 + og:type이 website 아님 + 작성 시각, 셋 다 있어야 기사로 인정
    # (작성자 프로필, 시세·카테고리 페이지, 기업 소개 페이지 등 제외)
    if (
        not features["ld_has_article"]
        or features["og_type"] == "website"
        or not features["page_published_at_et"]
    ):
        return "NOT_ARTICLE"

    # 언론사 작성일(동부시간)이 검색 대상일과 다르면 제외 (오래된 기사 재노출, 전날 기사 등)
    page_date = datetime.fromisoformat(features["page_published_at_et"]).date().isoformat()

    if page_date != target_date:
        return "OUT_OF_WINDOW"

    if not features["body"]:
        return "EXTRACT_FAILED"

    # 본문에 유료 안내 문구가 있으면 유료 기사
    if features["paywall_phrase"]:
        return "PAYWALL"

    # 문단 수는 trafilatura 줄바꿈이 사이트마다 달라 판정에 쓰지 않음 (참고용 컬럼)
    body_is_short = features["body_words"] < MIN_BODY_WORDS
    body_is_partial = (
        features["ld_coverage"] is not None
        and features["ld_coverage"] < MIN_LD_COVERAGE
    )

    # JSON-LD isAccessibleForFree=false여도 HTML에는 본문 전체가 있는 경우가 많음
    # (CNN, CNBC, Fortune 등은 브라우저에서만 가림)
    # → 본문이 짧거나 일부만 있을 때만 유료 기사로 판정 (ld_is_free는 참고용 컬럼)
    if features["ld_is_free"] is False and (body_is_short or body_is_partial):
        return "PAYWALL"

    if body_is_short:
        return "TOO_SHORT"

    if body_is_partial:
        return "PARTIAL_BODY"

    return "SUCCESS"


def analyze_samples(sample_df, fetch_map):
    rows = []

    for article in sample_df.to_dict("records"):
        url = article["original_url"]
        fetch = fetch_map.get(url)

        raw_html = read_cached_html(url) if fetch and fetch["html_saved"] else None

        features = None
        analyze_error = None

        if raw_html:
            try:
                features = extract_page_features(raw_html, fetch["final_url"])

            except Exception as e:
                analyze_error = f"{type(e).__name__}: {e}"

        status = classify_page(article["source"], article["target_date"], fetch, features)

        # RSS 발행 시각 − 언론사 작성 시각 (시간 단위, 양수면 RSS가 더 늦음)
        rss_vs_page_hours = None

        if features and features["page_published_at_et"]:
            gap = (
                datetime.fromisoformat(article["published_at_et"])
                - datetime.fromisoformat(features["page_published_at_et"])
            )
            rss_vs_page_hours = round(gap.total_seconds() / 3600, 2)

        if analyze_error:
            status = "ANALYZE_ERROR"

        rows.append({
            "source": article["source"],
            "source_name": CANDIDATE_SOURCES.get(article["source"]),
            "query": article["query"],
            "title": article["title"],
            "target_date": article["target_date"],
            "published_at_et": article["published_at_et"],  # Google News RSS 기준 발행 시각
            "original_url": url,
            "final_url": fetch["final_url"] if fetch else None,
            "http_status": fetch["http_status"] if fetch else None,
            "status": status,
            "analyze_error": analyze_error,
            **(features or {}),
            "rss_vs_page_hours": rss_vs_page_hours,
        })

    return pd.DataFrame(rows)

In [12]:
page_df = analyze_samples(sample_df, fetch_map)
page_df.to_parquet(PAGE_RESULT_PATH, index=False)

print(f"저장: {PAGE_RESULT_PATH} ({len(page_df)}건)")

pd.crosstab(page_df["source"], page_df["status"], margins=True)

저장: source_test/round2/page_results.parquet (600건)


status,EXCLUDED_SECTION,LIVE_BLOG,NOT_ARTICLE,NOT_HTML,OUT_OF_WINDOW,SUCCESS,TOO_SHORT,VIDEO_PAGE,All
source,,,,,,,,,
businessinsider.com,49,0,3,0,1,7,0,0,60
cbsnews.com,0,3,1,0,2,26,0,28,60
cnbc.com,5,0,3,0,1,42,0,9,60
cnn.com,34,0,1,0,0,15,0,10,60
fool.com,0,0,1,0,0,57,0,2,60
fortune.com,0,0,3,1,0,52,0,4,60
foxbusiness.com,0,0,1,0,0,13,0,46,60
nypost.com,0,0,2,0,1,57,0,0,60
theguardian.com,0,0,16,0,10,33,1,0,60


In [13]:
# 언론사별 작성 시각 추출률과 RSS 시각과의 차이
date_check_df = (
    page_df
    .assign(
        has_page_published=lambda d: d["page_published_at_et"].notna(),
        has_page_modified=lambda d: d["page_modified_at_et"].notna(),
        abs_gap_hours=lambda d: d["rss_vs_page_hours"].abs(),
    )
    .groupby("source")
    .agg(
        pages=("status", "size"),
        published_found=("has_page_published", "mean"),
        modified_found=("has_page_modified", "mean"),
        tz_assumed=("page_date_tz_assumed", "sum"),
        median_gap_hours=("rss_vs_page_hours", "median"),
        max_abs_gap_hours=("abs_gap_hours", "max"),
        over_1h=("abs_gap_hours", lambda x: (x > 1).sum()),
    )
    .round(2)
    .sort_values("published_found")
)

date_check_df

,pages,published_found,modified_found,tz_assumed,median_gap_hours,max_abs_gap_hours,over_1h
source,,,,,,,
foxbusiness.com,60,0.22,0.22,0,-8.33,19.00,12
cnn.com,60,0.35,0.35,0,-4.59,17.04,16
cbsnews.com,60,0.52,0.52,0,-10.38,11350.00,27
usatoday.com,60,0.72,0.72,0,-3.59,67.26,39
theguardian.com,60,0.73,0.73,0,-0.35,135006.65,35
fortune.com,60,0.87,0.87,0,-4.24,14.98,35
businessinsider.com,60,0.90,0.83,0,-6.00,534.92,34
cnbc.com,60,0.97,0.97,0,-6.16,85.57,46
fool.com,60,0.97,0.97,0,-5.78,20.89,49


## 5. 언론사별 요약 + 판정

In [14]:
def summarize_sources(page_df, rss_stats_df):
    counts = pd.crosstab(page_df["source"], page_df["status"])

    summary = (
        rss_stats_df
        .set_index("source")
        .join(page_df.groupby("source").size().rename("sampled"))
        .join(counts)
        .fillna(0)
    )

    status_cols = list(counts.columns)
    summary[status_cols + ["sampled"]] = summary[status_cols + ["sampled"]].astype(int)

    success = summary.get("SUCCESS", 0)
    filterable = sum(summary.get(s, 0) for s in FILTERABLE_STATUSES)

    summary["article_pages"] = summary["sampled"] - filterable
    summary["filterable_share"] = (filterable / summary["sampled"]).where(summary["sampled"] > 0)
    summary["video_share"] = (
        sum(summary.get(s, 0) for s in VIDEO_STATUSES) / summary["sampled"]
    ).where(summary["sampled"] > 0)
    summary["success_rate"] = (success / summary["sampled"]).where(summary["sampled"] > 0)
    summary["article_success_rate"] = (success / summary["article_pages"]).where(summary["article_pages"] > 0)

    summary["median_words"] = (
        page_df[page_df["status"] == "SUCCESS"]
        .groupby("source")["body_words"]
        .median()
    )

    # 성공이 아닌 페이지 중 가장 많은 실패 사유
    failures = page_df[~page_df["status"].isin({"SUCCESS"} | FILTERABLE_STATUSES)]
    summary["main_fail_reason"] = failures.groupby("source")["status"].agg(
        lambda x: x.value_counts().index[0]
    )

    def verdict(row):
        if row["rss_articles"] == 0:
            return "NO_RSS_RESULT"

        if row["article_pages"] < MIN_ARTICLE_PAGES:
            return "INSUFFICIENT_SAMPLE"

        if row["article_success_rate"] >= USABLE_RATE:
            return "USABLE"

        if row["article_success_rate"] >= UNSTABLE_RATE:
            return "UNSTABLE"

        return "NOT_USABLE"

    summary["verdict"] = summary.apply(verdict, axis=1)
    summary.insert(0, "source_name", summary.index.map(CANDIDATE_SOURCES))

    summary["verdict"] = pd.Categorical(summary["verdict"], categories=VERDICT_ORDER, ordered=True)

    return (
        summary
        .reset_index()
        .sort_values(["verdict", "article_success_rate"], ascending=[True, False])
        .reset_index(drop=True)
    )

In [15]:
source_summary = summarize_sources(page_df, rss_stats_df)
source_summary.to_csv(SOURCE_SUMMARY_PATH, index=False, encoding="utf-8-sig")

source_summary[[
    "source",
    "source_name",
    "verdict",
    "rss_articles",
    "sampled",
    "article_pages",
    "article_success_rate",
    "filterable_share",
    "median_words",
    "main_fail_reason",
]]

,source,source_name,verdict,rss_articles,sampled,article_pages,article_success_rate,filterable_share,median_words,main_fail_reason
0,cnbc.com,CNBC,USABLE,266,60,42,1.000000,0.300000,572.0,NaN
1,cbsnews.com,CBS News,USABLE,165,60,26,1.000000,0.566667,540.5,NaN
2,nypost.com,New York Post,USABLE,65,60,57,1.000000,0.050000,623.0,NaN
3,fool.com,The Motley Fool,USABLE,555,60,57,1.000000,0.050000,705.0,NaN
4,fortune.com,Fortune,USABLE,214,60,53,0.981132,0.116667,1074.5,NOT_HTML
5,theguardian.com,The Guardian,USABLE,164,60,34,0.970588,0.433333,720.0,TOO_SHORT
6,usatoday.com,USA Today,USABLE,118,60,22,0.954545,0.633333,652.0,TOO_SHORT
7,businessinsider.com,Business Insider,INSUFFICIENT_SAMPLE,605,60,7,1.000000,0.883333,770.0,NaN
8,cnn.com,CNN Business,INSUFFICIENT_SAMPLE,139,60,15,1.000000,0.750000,1089.0,NaN
9,foxbusiness.com,Fox Business,INSUFFICIENT_SAMPLE,165,60,13,1.000000,0.783333,488.0,NaN


In [ ]:
# ── 변수 정리: 최종 판정 ──
def sources_where(mask):
    return to_source_dict(source_summary.loc[mask, "source"])


USABLE_SOURCES = sources_where(source_summary["verdict"] == "USABLE")
UNSTABLE_SOURCES = sources_where(source_summary["verdict"] == "UNSTABLE")
NOT_USABLE_SOURCES = sources_where(source_summary["verdict"] == "NOT_USABLE")
INSUFFICIENT_SOURCES = sources_where(source_summary["verdict"] == "INSUFFICIENT_SAMPLE")
NO_RSS_SOURCES = sources_where(source_summary["verdict"] == "NO_RSS_RESULT")

# 영상/라이브 페이지 비중이 높은 언론사 (수집 시 URL 필터 필수)
VIDEO_HEAVY_SOURCES = sources_where(source_summary["video_share"] >= VIDEO_HEAVY_SHARE)

# 언론사별 주 실패 사유 {도메인: 상태값}
FAIL_REASON_BY_SOURCE = (
    source_summary
    .dropna(subset=["main_fail_reason"])
    .set_index("source")["main_fail_reason"]
    .to_dict()
)

source_lists = {
    "created_at": now_utc(),
    "test_queries": TEST_QUERIES,
    "test_dates": TEST_DATES,
    "USABLE": USABLE_SOURCES,
    "UNSTABLE": UNSTABLE_SOURCES,
    "NOT_USABLE": NOT_USABLE_SOURCES,
    "INSUFFICIENT_SAMPLE": INSUFFICIENT_SOURCES,
    "NO_RSS_RESULT": NO_RSS_SOURCES,
    "VIDEO_HEAVY": VIDEO_HEAVY_SOURCES,
    "FAIL_REASON": FAIL_REASON_BY_SOURCE,
}

SOURCE_LISTS_PATH.write_text(json.dumps(source_lists, ensure_ascii=False, indent=2))

for key in ["USABLE", "UNSTABLE", "NOT_USABLE", "INSUFFICIENT_SAMPLE", "NO_RSS_RESULT", "VIDEO_HEAVY"]:
    print(f"{key:20s}: {list(source_lists[key].values())}")

USABLE              : ['CNBC', 'Fortune', 'CBS News', 'New York Post', 'USA Today', 'The Guardian', 'The Motley Fool']
UNSTABLE            : []
NOT_USABLE          : []
INSUFFICIENT_SAMPLE : ['Business Insider', 'CNN Business', 'Fox Business']
NO_RSS_RESULT       : []
VIDEO_HEAVY         : ['Fox Business', 'CBS News']


## 6. 눈으로 검수하기

판정이 실제와 맞는지 확인용입니다. 특히 `SUCCESS`인데 영상 클립 설명문만 있는 경우,
`TOO_SHORT`인데 실제로는 짧은 정상 기사인 경우를 확인하고 0. 설정의 기준을 조정하세요.

In [17]:
def show_samples(source, status=None, n=3, chars=800):
    rows = page_df[page_df["source"] == source]

    if status:
        rows = rows[rows["status"] == status]

    for row in rows.head(n).to_dict("records"):
        print(f"[{row['status']}] {row['title']}")
        print(f"URL: {row['final_url'] or row['original_url']}")
        print(
            f"단어 {row.get('body_words')} | 문단 {row.get('body_paragraphs')} | "
            f"og:type {row.get('og_type')} | LD {row.get('ld_types')}"
        )
        print(str(row.get("body") or "")[:chars])
        print("\n" + "=" * 80 + "\n")


# 언론사 × 상태별 최대 3건씩 검수용 CSV
review_df = (
    page_df
    .groupby(["source", "status"], group_keys=False)
    .head(3)
    .assign(body_head=lambda d: d["body"].fillna("").str[:500])
    [["source", "status", "title", "final_url", "body_words", "body_paragraphs", "og_type", "body_head"]]
    .sort_values(["source", "status"])
)
review_df.to_csv(REVIEW_SAMPLE_PATH, index=False, encoding="utf-8-sig")

print(f"저장: {REVIEW_SAMPLE_PATH} ({len(review_df)}건)")

저장: source_test/round2/review_samples.csv (99건)


In [18]:
show_samples("cnbc.com", status="SUCCESS", n=2)

[SUCCESS] Cramer’s week ahead: What to watch as Wall Street navigates a tough September - CNBC
URL: https://www.cnbc.com/2026/09/18/jim-cramer-week-ahead-costco-okta.html
단어 513.0 | 문단 12.0 | og:type article | LD ['NewsArticle']
CNBC's Jim Cramer on Friday said investors could be in for a relatively quiet week, with only a handful of major earnings reports and corporate events on the calendar as Wall Street works its way through a historically difficult September.
"Remember, September is the cruelest month," the "Mad Money" host said. "It's been okay — not-great — so far. … Here's hoping we'll have a relatively sedate couple of weeks."
The Dow Jones Industrial Average bore the brunt of the Federal Reserve's quarter-point interest rate hike Wednesday. The blue-chip index fell 1.7% for the week as policymakers signaled that additional tightening could be ahead. The S&P 500 dropped 0.1%. Meanwhile, the tech-heavy Nasdaq Composite held up better, rising 0.7% as investors returned to the AI

In [19]:
show_samples("cnbc.com", status="VIDEO_PAGE", n=2)

[VIDEO_PAGE] AGCO CEO Eric Hansotia on the tariff impacts on AG industry - CNBC
URL: https://www.cnbc.com/video/2026/09/08/agco-ceo-eric-hansotia-on-the-tariff-impacts-on-ag-industry.html
단어 45.0 | 문단 1.0 | og:type video | LD ['VideoObject']
watch now
AGCO CEO Eric Hansotia on the tariff impacts on AG industry
AGCO CEO Eric Hansotia joins 'The Exchange' to discuss the impact of Canada's tariffs on the agricultural industry, new developments in farming technology and more.
02:52
Tue, Sep 8 20261:50 PM EDT


[VIDEO_PAGE] Stock futures inch lower after S&P 500 hits fresh record - CNBC
URL: https://www.cnbc.com/video/2026/10/07/stock-futures-inch-lower-after-sp-500-hits-fresh-record.html
단어 0.0 | 문단 0.0 | og:type video | LD ['VideoObject']





In [ ]:
show_samples("theguardian.com", status="TOO_SHORT", n=2)

[TOO_SHORT] ‘We don’t have inflation’? Five charts show the US economic reality of Trump’s second term so far - The Guardian
URL: https://www.theguardian.com/us-news/2026/oct/26/trump-second-term-economy-charts
단어 48.0 | 문단 1.0 | og:type article | LD ['NewsArticle', 'WebPage']
In Trump’s view, his plan is working. At a recent rally in Montana, Trump boasted about the current economy, framing it as a booming time for Americans. “We’re richer now,” he said. “We don’t have inflation.”
Here’s what the data shows us about Trump’s second term so far.




In [20]:
source_summary

,source,source_name,rss_requests,rss_errors,rss_truncated,rss_hits,rss_articles,sampled,EXCLUDED_SECTION,LIVE_BLOG,...,TOO_SHORT,VIDEO_PAGE,article_pages,filterable_share,video_share,success_rate,article_success_rate,median_words,main_fail_reason,verdict
0,cnbc.com,CNBC,72,0,0,451,266,60,5,0,...,0,9,42,0.300000,0.150000,0.700000,1.000000,572.0,NaN,USABLE
1,cbsnews.com,CBS News,72,0,0,188,165,60,0,3,...,0,28,26,0.566667,0.516667,0.433333,1.000000,540.5,NaN,USABLE
2,nypost.com,New York Post,72,0,0,95,65,60,0,0,...,0,0,57,0.050000,0.000000,0.950000,1.000000,623.0,NaN,USABLE
3,fool.com,The Motley Fool,72,0,7,942,555,60,0,0,...,0,2,57,0.050000,0.033333,0.950000,1.000000,705.0,NaN,USABLE
4,fortune.com,Fortune,72,0,0,375,214,60,0,0,...,0,4,53,0.116667,0.066667,0.866667,0.981132,1074.5,NOT_HTML,USABLE
5,theguardian.com,The Guardian,72,0,0,192,164,60,0,0,...,1,0,34,0.433333,0.000000,0.550000,0.970588,720.0,TOO_SHORT,USABLE
6,usatoday.com,USA Today,72,0,0,148,118,60,22,0,...,1,6,22,0.633333,0.100000,0.350000,0.954545,652.0,TOO_SHORT,USABLE
7,businessinsider.com,Business Insider,72,0,8,848,605,60,49,0,...,0,0,7,0.883333,0.000000,0.116667,1.000000,770.0,NaN,INSUFFICIENT_SAMPLE
8,cnn.com,CNN Business,72,0,0,227,139,60,34,0,...,0,10,15,0.750000,0.166667,0.250000,1.000000,1089.0,NaN,INSUFFICIENT_SAMPLE
9,foxbusiness.com,Fox Business,72,0,0,194,165,60,0,0,...,0,46,13,0.783333,0.766667,0.216667,1.000000,488.0,NaN,INSUFFICIENT_SAMPLE
